In [1]:
# ============================================================
# NOMBRE D'HÔPITAUX À MOINS DE 100 SÉJOURS (eICU-CRD v2.0)
# Cellule autonome : mêmes critères que l'étape 1 du notebook 01
# ============================================================
from pathlib import Path
import pandas as pd

# ---------- À adapter ----------
EICU_DIR = Path("../../../dataset")   # dossier des fichiers eICU-CRD

# ---------- Paramètres (identiques au notebook 01) ----------
MIN_AGE = 18
MIN_ICU_MIN = 1440          # durée en unité >= 24 h
MIN_STAYS_LOCKED = 100      # seuil : < 100 séjours -> exclu
MIN_STAYS_DEV = 500         # >= 500 -> développement

def table_path(name):
    for ext in (".csv.gz", ".csv"):
        p = EICU_DIR / f"{name}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Table '{name}' introuvable dans {EICU_DIR.resolve()}")

# ---------- Lecture de la table patient ----------
cols = ["patientunitstayid", "uniquepid", "patienthealthsystemstayid", "hospitalid",
        "unitvisitnumber", "age", "unitdischargeoffset", "hospitaldischargestatus",
        "hospitaldischargeyear"]
patient = pd.read_csv(table_path("patient"), usecols=cols, low_memory=False)

p = patient.copy()
p["age_num"] = pd.to_numeric(p["age"].astype(str).str.replace("> 89", "90", regex=False),
                             errors="coerce")
all_hospitals = set(p["hospitalid"].unique())

# ---------- Critères d'éligibilité (ordre de l'étape 1) ----------
flow = [("Séjours en unité (eICU)", len(p), p["uniquepid"].nunique(), p["hospitalid"].nunique())]
def keep(df, label):
    flow.append((label, len(df), df["uniquepid"].nunique(), df["hospitalid"].nunique()))
    return df

p = keep(p[p["unitvisitnumber"] == 1], "1re unité du séjour hospitalier")
p = keep(p[p["age_num"] >= MIN_AGE], f"Âge >= {MIN_AGE} ans")
p = keep(p[p["unitdischargeoffset"] >= MIN_ICU_MIN], "Durée en unité >= 24 h")
p = keep(p[p["hospitaldischargestatus"].isin(["Alive", "Expired"])], "Statut de sortie connu")
p = p.sort_values(["uniquepid", "hospitaldischargeyear", "patienthealthsystemstayid",
                   "patientunitstayid"], na_position="last")
p = keep(p.drop_duplicates("uniquepid", keep="first"), "1er séjour éligible de chaque patient")

# ---------- Taille des hôpitaux ----------
sizes = p.groupby("hospitalid").size()
hosp_no_eligible = all_hospitals - set(sizes.index)   # hôpitaux sans aucun séjour éligible

small  = sizes[sizes < MIN_STAYS_LOCKED]
locked = sizes[(sizes >= MIN_STAYS_LOCKED) & (sizes < MIN_STAYS_DEV)]
dev    = sizes[sizes >= MIN_STAYS_DEV]

n_small_with_stays = len(small)
n_small_total = n_small_with_stays + len(hosp_no_eligible)

# ---------- Résultats ----------
print(pd.DataFrame(flow, columns=["Étape", "Séjours", "Patients", "Hôpitaux"]).to_string(index=False))
print()
print(f"Hôpitaux dans eICU (table patient)               : {len(all_hospitals)}")
print(f"Hôpitaux sans aucun séjour éligible              : {len(hosp_no_eligible)}")
print(f"Hôpitaux avec 1 à {MIN_STAYS_LOCKED - 1} séjours éligibles        : {n_small_with_stays}  ({int(small.sum())} séjours)")
print(f"Hôpitaux de {MIN_STAYS_LOCKED} à {MIN_STAYS_DEV - 1} séjours (test verrouillé) : {len(locked)}  ({int(locked.sum())} séjours)")
print(f"Hôpitaux >= {MIN_STAYS_DEV} séjours (développement)     : {len(dev)}  ({int(dev.sum())} séjours)")
print()
print(f"==> NOMBRE D'HÔPITAUX À MOINS DE {MIN_STAYS_LOCKED} SÉJOURS : {n_small_total}")
print(f"    (dont {n_small_with_stays} avec au moins 1 séjour éligible)")

# Détail des petits hôpitaux
detail = small.sort_values().rename("n_sejours_eligibles").reset_index()
display(detail)
display(sizes.describe().round(1))

                                Étape  Séjours  Patients  Hôpitaux
              Séjours en unité (eICU)   200859    139367       208
      1re unité du séjour hospitalier   158442    133492       208
                        Âge >= 18 ans   157883    132994       208
               Durée en unité >= 24 h   110257     95823       208
               Statut de sortie connu   109218     95012       207
1er séjour éligible de chaque patient    95012     95012       207

Hôpitaux dans eICU (table patient)               : 208
Hôpitaux sans aucun séjour éligible              : 1
Hôpitaux avec 1 à 99 séjours éligibles        : 49  (1912 séjours)
Hôpitaux de 100 à 499 séjours (test verrouillé) : 97  (24796 séjours)
Hôpitaux >= 500 séjours (développement)     : 61  (68304 séjours)

==> NOMBRE D'HÔPITAUX À MOINS DE 100 SÉJOURS : 50
    (dont 49 avec au moins 1 séjour éligible)


,hospitalid,n_sejours_eligibles
0,385,2
1,136,2
2,156,4
3,151,4
4,86,4
5,90,4
6,94,5
7,409,6
8,265,6
9,91,7


count     207.0
mean      459.0
std       571.6
min         2.0
25%       105.0
50%       260.0
75%       571.5
max      3539.0
dtype: float64

In [2]:
%run ../src/test_opcal_gp.py

OK    T1 retour à la moyenne : t=0: |ã−μ̂|=7.23e-01 ; t=2: |ã−μ̂|=7.99e-01 ; t=5: |ã−μ̂|=4.88e-01 ; t=20: |ã−μ̂|=4.34e-05 ; t=100: |ã−μ̂|=0.00e+00 ; convergé=True
OK    T2 cas p = 1 contre calibration de régression : écart relatif max 0.025 ; pente estimée 0.520 (vraie 0,5 ; atténuée par le bruit des entrées d'entraînement) ; ẑ=1.5, C=0.05, λ=0.95: GP 0.724 / RC 0.737 ; ẑ=1.5, C=1, λ=0.48: GP 0.350 / RC 0.356 ; ẑ=-1, C=2, λ=0.32: GP -0.196 / RC -0.192 ; ẑ=0.8, C=0.3, λ=0.76: GP 0.301 / RC 0.304
OK    T2b dé-atténuation (p = 10, règle par défaut) : pente vraie 0.25 ; avec débruitage 0.264 ; sans 0.185 ; MCO naïfs 0.202
OK    T3 positivité et forme exacte : valeur propre min 6.76e-01 ; exacte 0.14068 contre Monte Carlo 0.14041 ± 0.00026
TOUS LES TESTS RÉUSSIS
